<a href="https://colab.research.google.com/github/manuelcdr/JumbaZ/blob/master/Gemma4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import os
import subprocess
import time
import urllib.request

# Must match GEMMA_MODEL in claude_desktop_config.json (mcpServers.gemma.env)
# Gemma 4 in Ollama is gemma4:* NOT gemma3:* (different downloads)
OLLAMA_MODEL = "gemma4:latest"

os.environ["DEBIAN_FRONTEND"] = "noninteractive"


def run_sh(cmd, label):
    print(label, flush=True)
    subprocess.run(
        cmd,
        shell=True,
        check=True,
        stdin=subprocess.DEVNULL,
    )


run_sh(
    "curl -fsSL https://ollama.com/install.sh | sh",
    "Installing Ollama (often 2-8 min; no output from the installer is normal)...",
)

print("Starting ollama serve in background...", flush=True)
subprocess.Popen(
    "nohup ollama serve > /tmp/ollama.log 2>&1 &",
    shell=True,
    stdin=subprocess.DEVNULL,
)

print("Waiting for http://127.0.0.1:11434 ...", flush=True)
for i in range(90):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("Ollama API is up.", flush=True)
        break
    except Exception:
        if i % 10 == 0 and i > 0:
            print(
                f"  ...still waiting ({i * 2}s). If this never ends, open a new cell and run: !cat /tmp/ollama.log",
                flush=True,
            )
        time.sleep(2)
else:
    raise RuntimeError("Ollama API did not start. In a new cell run: !cat /tmp/ollama.log")

ls = subprocess.run(["ollama", "list"], capture_output=True, text=True, check=True)
if OLLAMA_MODEL in ls.stdout:
    print(f"Skip pull: {OLLAMA_MODEL} already installed.", flush=True)
else:
    print(
        f"Pulling {OLLAMA_MODEL} (large download; can take 10-40+ min on Colab)...",
        flush=True,
    )
    subprocess.run(
        ["ollama", "pull", OLLAMA_MODEL],
        check=True,
        stdin=subprocess.DEVNULL,
    )
print(f"Done: {OLLAMA_MODEL} ready on http://127.0.0.1:11434", flush=True)

Installing Ollama (often 2-8 min; no output from the installer is normal)...
Starting ollama serve in background...
Waiting for http://127.0.0.1:11434 ...
Ollama API is up.
Skip pull: gemma4:latest already installed.
Done: gemma4:latest ready on http://127.0.0.1:11434


In [22]:
# Install the tunnel tool
!pip install pyngrok

from pyngrok import ngrok
import os

# 1. Plug in your secret key
ngrok.set_auth_token("3CWikD8OQwKNxOTIs3FJ1SSHGZK_7cbupDcUyC4ZCDNWT3B9f")

# 2. Open the tunnel to the Ollama port (11434)
# We use 'http' because Ollama talks in web-style messages
public_url = ngrok.connect(11434, "http").public_url

print("---------------------------------------------------------")
print(f"YOUR PUBLIC ENDPOINT IS: {public_url}")
print("---------------------------------------------------------")
print("KEEP THIS TAB OPEN! If you close Colab, the tunnel breaks.")


---------------------------------------------------------
YOUR PUBLIC ENDPOINT IS: https://daintily-gimmick-overboard.ngrok-free.dev
---------------------------------------------------------
KEEP THIS TAB OPEN! If you close Colab, the tunnel breaks.


In [23]:
import requests

APPS_SCRIPT_URL = "https://script.google.com/macros/s/AKfycbybvcNg_TTqh169JgaAMPRI2YaHVczHqNcMVvycvRptXkrSDMuXokvr08f4qLVFTTZfgw/exec"

try:
    t = requests.get("http://127.0.0.1:4040/api/tunnels", timeout=10)
    t.raise_for_status()
    tunnels = t.json().get("tunnels") or []
    if not tunnels:
        raise RuntimeError("No ngrok tunnels - start ngrok first.")

    public = None
    for tunnel in tunnels:
        u = tunnel.get("public_url") or ""
        if u.startswith("https://"):
            public = u
            break
    if not public:
        public = tunnels[0]["public_url"]

    p = requests.post(APPS_SCRIPT_URL, json={"url": public}, timeout=30)
    p.raise_for_status()
    print("OK Gemma registered at:", public)
except Exception as e:
    print("Registration failed:", e)
    print("Fix: run the ngrok cell first, then run this cell again.")

OK Gemma registered at: https://daintily-gimmick-overboard.ngrok-free.dev
